# Lab configuration (shared)

Every lab notebook loads this with `%run ./_config`.

**Widgets at the top of your notebook** (after you run this cell once) control catalog, schema, model, and warehouse. Change any widget anytime, then re-run from this `%run` cell downward.

You do **not** need to be a workspace admin. Your instructor grants catalog/schema rights, warehouse `CAN USE`, and model access:

- **Preferred:** `EXECUTE` on a Unity Catalog model service (for example `system.ai.<model>`), plus `USE CATALOG` on `system` and `USE SCHEMA` on `system.ai`
- **Fallback:** `CAN QUERY` on a legacy Foundation Model **serving endpoint** (workshop default name below)

### How to set `llm_endpoint`

Paste **either**:

1. A UC model service FQN such as `system.ai.<model-name>`, **or**
2. A legacy serving endpoint name such as `databricks-meta-llama-3-3-70b-instruct`

If preflight says Unity Gateway / model service is unavailable, use the **legacy endpoint name your admin gave you**. That fallback is normal when Gateway is not enabled in the workspace.

### Docs
- [Notebook widgets (Azure Databricks)](https://learn.microsoft.com/en-us/azure/databricks/notebooks/widgets)
- [Unity Catalog overview](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/)
- [Migrate to Unity Gateway](https://docs.databricks.com/aws/en/ai-gateway/migrate-to-unity-gateway)


## Read widgets into Python variables

Edit the widget values in the bar above the notebook (not in this cell). Blank **schema** means auto-name from your login.

Widget **3** accepts a UC model service FQN (`system.ai…`) or a legacy endpoint name.


In [ ]:
import os
import re

# Text widgets: change anytime in the widget bar, then re-run this notebook from %run ./_config
dbutils.widgets.text("catalog", "main", "1. Catalog")
dbutils.widgets.text("schema", "ai_literacy_2a", "2. Schema (blank = auto from login)")
dbutils.widgets.text(
    "llm_endpoint",
    "databricks-meta-llama-3-3-70b-instruct",
    "3. LLM model (UC FQN or legacy endpoint)",
)
dbutils.widgets.text("warehouse_name", "Serverless Starter Warehouse", "4. SQL warehouse")
dbutils.widgets.text("demo_filter_state", "CA", "6. State hidden by row filter")
dbutils.widgets.text("pii_group", "lab_pii_viewers", "7. Group that sees unmasked PII")

# Who is running the notebook (agent tools inherit this identity)
USER = spark.sql("SELECT current_user()").first()[0]

# Catalog / schema from widgets (strip spaces; auto-schema if blank)
CATALOG = dbutils.widgets.get("catalog").strip()
_schema_raw = dbutils.widgets.get("schema").strip()
SCHEMA = _schema_raw or ("lab_" + re.sub(r"[^a-z0-9]", "_", USER.split("@")[0].lower()))

# Model + warehouse names for Labs 2–5 / Genie
# Default is a legacy serving endpoint (works when Unity Gateway is not enforced).
# Paste a system.ai.<model> FQN to use Unity Gateway instead.
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint").strip()
LEGACY_LLM_FALLBACK = "databricks-meta-llama-3-3-70b-instruct"
WAREHOUSE_NAME = dbutils.widgets.get("warehouse_name").strip()

# Governance knobs used when Lab 1 creates filter/mask
HIDDEN_STATE = dbutils.widgets.get("demo_filter_state").strip() or "CA"
PII_GROUP = dbutils.widgets.get("pii_group").strip() or "lab_pii_viewers"

# SQL helpers: quoted for Spark SQL, plain for UI / APIs / Playground
FQ = f"`{CATALOG}`.`{SCHEMA}`"
FQ_PLAIN = f"{CATALOG}.{SCHEMA}"

# MLflow experiment lives under your home folder (no admin needed)
EXPERIMENT_PATH = f"/Users/{USER}/ai_literacy_agent_eval"


def is_uc_model_service(name: str) -> bool:
    """True if the widget looks like a UC model service FQN (e.g. system.ai.claude-sonnet-4-5)."""
    n = (name or "").strip()
    return "." in n and not n.startswith("databricks-")


def _workspace_host() -> str:
    from databricks.sdk import WorkspaceClient
    host = WorkspaceClient().config.host
    return host.rstrip("/")


def _notebook_api_token() -> str:
    """Token for notebook-side HTTP clients (Gateway OpenAI client)."""
    try:
        return (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .apiToken()
            .get()
        )
    except Exception:
        return os.environ.get("DATABRICKS_TOKEN", "")


def get_llm_client():
    """OpenAI-compatible client: Unity Gateway for UC FQNs, legacy serving endpoints otherwise."""
    from openai import OpenAI
    from databricks.sdk import WorkspaceClient

    w = WorkspaceClient()
    if is_uc_model_service(LLM_ENDPOINT):
        return OpenAI(
            api_key=_notebook_api_token(),
            base_url=f"{_workspace_host()}/ai-gateway/mlflow/v1",
        )
    return w.serving_endpoints.get_open_ai_client()


def resolve_judge_model():
    """MLflow judge URI. Prefer the widget model; fall back to legacy if FQN judges fail later."""
    if is_uc_model_service(LLM_ENDPOINT):
        return f"databricks:/{LLM_ENDPOINT}"
    return f"databricks:/{LLM_ENDPOINT}"


JUDGE_MODEL = resolve_judge_model()
JUDGE_MODEL_LEGACY_FALLBACK = f"databricks:/{LEGACY_LLM_FALLBACK}"
LLM_ACCESS_MODE = "unity_gateway" if is_uc_model_service(LLM_ENDPOINT) else "legacy_endpoint"


def probe_llm_access():
    """
    Preflight for the model widget.
    Returns (ok: bool, detail: str, mode: str).
    """
    from databricks.sdk import WorkspaceClient

    w = WorkspaceClient()
    name = LLM_ENDPOINT
    if not name:
        return False, "llm_endpoint widget is empty", "none"

    if is_uc_model_service(name):
        # Prefer a tiny Gateway completion; if Gateway is unavailable, tell user to use legacy.
        try:
            client = get_llm_client()
            client.chat.completions.create(
                model=name,
                messages=[{"role": "user", "content": "ping"}],
                max_tokens=8,
                temperature=0,
            )
            return True, f"Unity Gateway OK for model service {name}", "unity_gateway"
        except Exception as e:
            return (
                False,
                (
                    f"Unity Gateway / model service probe failed for {name!r}. "
                    f"This often means Unity Gateway is not enabled (or is still a preview) in this workspace. "
                    f"Set the llm_endpoint widget to the legacy serving endpoint your admin gave you "
                    f"(workshop default: {LEGACY_LLM_FALLBACK!r}) and ensure you have CAN QUERY on that endpoint. "
                    f"Error: {e}"
                ),
                "unity_gateway",
            )

    # Legacy serving endpoint
    try:
        ep = w.serving_endpoints.get(name)
        ready = getattr(getattr(ep, "state", None), "ready", None) or getattr(ep, "state", None)
        return True, f"Legacy endpoint {name} reachable (state={ready})", "legacy_endpoint"
    except Exception as e:
        return (
            False,
            (
                f"Cannot read serving endpoint {name!r}. "
                f"Ask for CAN QUERY on that endpoint, or paste a system.ai.<model> FQN if Unity Gateway is enabled. "
                f"({e})"
            ),
            "legacy_endpoint",
        )


def resolve_sql_warehouse():
    """Return the warehouse matching the widget, or list warehouses you can see."""
    from databricks.sdk import WorkspaceClient
    seen = []
    for x in WorkspaceClient().warehouses.list():
        st = getattr(getattr(x, "state", None), "value", None) or str(getattr(x, "state", "") or "")
        seen.append(f"{x.name!r} [{st}]")
        if x.name == WAREHOUSE_NAME:
            return x
    raise RuntimeError(
        f"No warehouse named {WAREHOUSE_NAME!r} among warehouses you can use: "
        f"{', '.join(seen) or '(none visible)'}. Change the warehouse_name widget, or ask the instructor "
        f"for CAN USE on {WAREHOUSE_NAME!r}."
    )


def _notebook_folder():
    """Absolute /Workspace/... folder containing the calling notebook."""
    path = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook()
        .getContext()
        .notebookPath()
        .get()
    )
    return "/Workspace" + path.rsplit("/", 1)[0]


def show_diagram(filename, caption=""):
    """Load a PNG from ./assets next to the notebook (workspace FILE, not notebook)."""
    from IPython.display import Image, display, Markdown
    folder = _notebook_folder()
    path = f"{folder}/assets/{filename}"
    if caption:
        display(Markdown(f"*{caption}*"))
    try:
        display(Image(filename=path))
        print(f"Diagram OK: {path}")
    except Exception as e:
        try:
            display(Image(filename=f"./assets/{filename}"))
            print(f"Diagram OK via ./assets/{filename}")
        except Exception as e2:
            raise FileNotFoundError(
                f"Could not load diagram {filename}. Expected workspace FILE at {path}. "
                f"Sync assets/ next to the notebooks. Errors: {e!r} / {e2!r}"
            ) from e2
